#### ============================================================
#### Fake News Detection - Final Project Code V4
#### Author: Xiaoxi Gao & Zhenzhe Luo
####
#### Goal:
#### Build a reproducible and interpretable machine learning system
#### to classify fake vs true news articles.
####
#### Standard pipeline:
#### 1. Load data
#### 2. Clean text
#### 3. Train/Test split
#### 4. TF-IDF vectorization
#### 5. Cross validation training
#### 6. Hyperparameter tuning (GridSearchCV)
#### 7. Model comparison (F1)
#### 8. Retrain best model on full training data
#### 9. Final evaluation on test set (F1, AUC, ROC, Accuracy)
####
#### Instructor-required components:
#### - baseline solution
#### - statistical component (bootstrapping)
#### - interpretable results
#### - ensemble learning (stacking)
#### ============================================================

In [2]:
import os
import re
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.dummy import DummyClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier, StackingClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    roc_curve,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

#### ============================================================
#### 0. Configuration
#### ============================================================

In [3]:
FAKE_PATH = "Fake.csv"
TRUE_PATH = "True.csv"

RANDOM_STATE = 42
TEST_SIZE = 0.20
N_SPLITS = 5
BOOTSTRAP_ROUNDS = 300

OUTPUT_DIR = "final_results_fake_news_v4"
os.makedirs(OUTPUT_DIR, exist_ok=True)

#### ============================================================
####  Utility functions
####  ============================================================

In [4]:
def print_section(title):
    print("\n" + "=" * 100)
    print(title)
    print("=" * 100)


def save_text_report(text, filename):
    with open(os.path.join(OUTPUT_DIR, filename), "w", encoding="utf-8") as f:
        f.write(text)


def clean_text(text):
    """
    Clean English news text.
    1. Remove URLs
    2. Remove HTML tags
    3. Lowercase
    4. Remove non-letter characters
    5. Remove extra spaces
    """
    text = str(text)
    text = re.sub(r"http\S+|www\S+|https\S+", " ", text)
    text = re.sub(r"<.*?>", " ", text)
    text = text.lower()
    text = re.sub(r"[^a-zA-Z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def build_text_column(df):
    """
    Combine subject + title + text if available.
    """
    title_col = None
    text_col = None
    subject_col = None

    for c in df.columns:
        lc = c.lower()
        if lc == "title":
            title_col = c
        elif lc == "text":
            text_col = c
        elif lc == "subject":
            subject_col = c

    parts = []
    if subject_col is not None:
        parts.append(df[subject_col].fillna("").astype(str))
    if title_col is not None:
        parts.append(df[title_col].fillna("").astype(str))
    if text_col is not None:
        parts.append(df[text_col].fillna("").astype(str))

    if len(parts) == 0:
        raise ValueError("No usable text columns found. Expected subject/title/text.")

    combined = parts[0]
    for p in parts[1:]:
        combined = combined + " " + p

    return combined


def bootstrap_metric_ci(y_true, y_pred, metric_func, n_rounds=300, random_state=42):
    """
    Bootstrap 95% confidence interval.
    This is the statistical component required by the instructor.
    """
    rng = np.random.default_rng(random_state)
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    n = len(y_true)

    scores = []
    for _ in range(n_rounds):
        idx = rng.integers(0, n, size=n)
        yt = y_true[idx]
        yp = y_pred[idx]
        scores.append(metric_func(yt, yp))

    mean_score = np.mean(scores)
    lower = np.percentile(scores, 2.5)
    upper = np.percentile(scores, 97.5)

    return mean_score, lower, upper


def get_model_scores(model, X):
    """
    Return continuous scores for AUC/ROC.
    Prefer decision_function, otherwise predict_proba.
    """
    if hasattr(model, "decision_function"):
        return model.decision_function(X)
    elif hasattr(model, "predict_proba"):
        return model.predict_proba(X)[:, 1]
    else:
        return None


#### ============================================================
#### STEP 1 - Load data
#### ============================================================

In [5]:
print_section("STEP 1 - Load Data")

fake_df = pd.read_csv(FAKE_PATH)
true_df = pd.read_csv(TRUE_PATH)

# Label definition: Fake = 0, True = 1
fake_df["label"] = 0
true_df["label"] = 1

# Build combined text
fake_df["combined_text"] = build_text_column(fake_df)
true_df["combined_text"] = build_text_column(true_df)

# Merge
df = pd.concat([fake_df, true_df], ignore_index=True)
df = df[["combined_text", "label"]].copy()

print("Original dataset shape:", df.shape)
print("Original label distribution:")
print(df["label"].value_counts())


STEP 1 - Load Data
Original dataset shape: (44898, 2)
Original label distribution:
label
0    23481
1    21417
Name: count, dtype: int64


#### ============================================================
#### STEP 2 - Clean text
#### ============================================================

In [6]:
print_section("STEP 2 - Clean Text")

df["combined_text"] = df["combined_text"].astype(str).apply(clean_text)

# Remove empty texts
df = df[df["combined_text"].str.len() > 0]

# Remove duplicates
df = df.drop_duplicates(subset=["combined_text"]).reset_index(drop=True)

print("Dataset shape after cleaning:", df.shape)
print("Label distribution after cleaning:")
print(df["label"].value_counts())


STEP 2 - Clean Text
Dataset shape after cleaning: (44673, 2)
Label distribution after cleaning:
label
0    23467
1    21206
Name: count, dtype: int64


#### ============================================================
#### STEP 3 - Train/Test split
#### ============================================================

In [7]:
print_section("STEP 3 - Train/Test Split")

X = df["combined_text"]
y = df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y
)

print("Training samples:", len(X_train))
print("Testing samples :", len(X_test))


STEP 3 - Train/Test Split
Training samples: 35738
Testing samples : 8935


#### ============================================================
#### STEP 4 - TF-IDF Vectorization
#### TF-IDF stays inside the pipeline to avoid data leakage.
#### ============================================================

In [8]:
print_section("STEP 4 - TF-IDF Vectorization")

print("TF-IDF is included inside each pipeline to avoid data leakage.")

tfidf = TfidfVectorizer(
    stop_words="english",
    lowercase=True,
    strip_accents="unicode",
    max_df=0.85,
    min_df=3,
    ngram_range=(1, 2),
    max_features=30000,
    sublinear_tf=True
)


STEP 4 - TF-IDF Vectorization
TF-IDF is included inside each pipeline to avoid data leakage.


#### ============================================================
#### STEP 5 - Cross validation training
#### STEP 6 - Hyperparameter tuning (GridSearchCV)
#### ============================================================

In [ ]:
print_section("STEP 5 & STEP 6 - Cross Validation + Hyperparameter Tuning")

cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

# ------------------------------------------------------------
# Model candidates
# Baseline_Dummy: baseline only
# RandomForest: tree-based model
# Stacking: ensemble model
# ------------------------------------------------------------
model_configs = {
    "Baseline_Dummy": {
        "model": DummyClassifier(strategy="most_frequent"),
        "param_grid": None
    },

    "MultinomialNB": {
        "model": MultinomialNB(),
        "param_grid": {
            "clf__alpha": [0.1, 0.5, 1.0]
        }
    },

    "LogisticRegression": {
        "model": LogisticRegression(max_iter=3000, random_state=RANDOM_STATE),
        "param_grid": {
            "clf__C": [0.1, 1.0, 10.0]
        }
    },

    "LinearSVC": {
        "model": LinearSVC(random_state=RANDOM_STATE),
        "param_grid": {
            "clf__C": [0.1, 1.0, 10.0]
        }
    },

    "RandomForest": {
        "model": RandomForestClassifier(
            random_state=RANDOM_STATE,
            n_jobs=-1
        ),
        "param_grid": {
            "clf__n_estimators": [100, 200],
            "clf__max_depth": [None, 20]
        }
    },

    "Stacking": {
        "model": StackingClassifier(
            estimators=[
                ("lr", LogisticRegression(C=1.0, max_iter=3000, random_state=RANDOM_STATE)),
                ("nb", MultinomialNB(alpha=0.5)),
                ("svm", LinearSVC(C=1.0, random_state=RANDOM_STATE)),
                ("rf", RandomForestClassifier(
                    n_estimators=100,
                    max_depth=None,
                    random_state=RANDOM_STATE,
                    n_jobs=-1
                ))
            ],
            final_estimator=LogisticRegression(max_iter=3000, random_state=RANDOM_STATE),
            cv=5,
            n_jobs=-1
        ),
        "param_grid": {
            "clf__final_estimator__C": [0.1, 1.0, 10.0]
        }
    }
}

results = []
best_estimators = {}

for model_name, config in model_configs.items():
    print(f"\n>>> Running model: {model_name}")

    pipeline = Pipeline([
        ("tfidf", tfidf),
        ("clf", config["model"])
    ])

    # Baseline model: no tuning
    if model_name == "Baseline_Dummy":
        pipeline.fit(X_train, y_train)
        y_pred_test = pipeline.predict(X_test)

        row = {
            "Model": model_name,
            "Best_Params": "N/A",
            "CV_F1_Mean": 0.0,
            "Test_Accuracy": accuracy_score(y_test, y_pred_test),
            "Test_Precision": precision_score(y_test, y_pred_test, zero_division=0),
            "Test_Recall": recall_score(y_test, y_pred_test, zero_division=0),
            "Test_F1": f1_score(y_test, y_pred_test, zero_division=0)
        }

        results.append(row)
        best_estimators[model_name] = pipeline

        print("Best Params:", row["Best_Params"])
        print("CV F1 Mean :", row["CV_F1_Mean"])
        print("Test F1    :", round(row["Test_F1"], 4))

    else:
        grid = GridSearchCV(
            estimator=pipeline,
            param_grid=config["param_grid"],
            scoring="f1",
            cv=cv,
            n_jobs=-1,
            refit=True
        )

        grid.fit(X_train, y_train)

        best_model = grid.best_estimator_
        y_pred_test = best_model.predict(X_test)

        row = {
            "Model": model_name,
            "Best_Params": grid.best_params_,
            "CV_F1_Mean": grid.best_score_,
            "Test_Accuracy": accuracy_score(y_test, y_pred_test),
            "Test_Precision": precision_score(y_test, y_pred_test, zero_division=0),
            "Test_Recall": recall_score(y_test, y_pred_test, zero_division=0),
            "Test_F1": f1_score(y_test, y_pred_test, zero_division=0)
        }

        results.append(row)
        best_estimators[model_name] = best_model

        print("Best Params:", grid.best_params_)
        print("CV F1 Mean :", round(grid.best_score_, 4))
        print("Test F1    :", round(row["Test_F1"], 4))


STEP 5 & STEP 6 - Cross Validation + Hyperparameter Tuning

>>> Running model: Baseline_Dummy
Best Params: N/A
CV F1 Mean : 0.0
Test F1    : 0.0

>>> Running model: MultinomialNB


/Users/mac/anaconda3/lib/python3.11/site-packages/pandas/core/arrays/masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (
/Users/mac/anaconda3/lib/python3.11/site-packages/pandas/core/arrays/masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (
/Users/mac/anaconda3/lib/python3.11/site-packages/pandas/core/arrays/masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (
/Users/mac/anaconda3/lib/python3.11/site-packages/pandas/core/arrays/masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (
/Users/mac/anaconda3/lib/python3.11/site-packages/pandas/core/arrays/masked.py:61: UserWarning: Pandas requires version 

Best Params: {'clf__alpha': 0.1}
CV F1 Mean : 0.9657
Test F1    : 0.9657

>>> Running model: LogisticRegression
Best Params: {'clf__C': 10.0}
CV F1 Mean : 0.9981
Test F1    : 0.9981

>>> Running model: LinearSVC


/Users/mac/anaconda3/lib/python3.11/site-packages/pandas/core/arrays/masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (
/Users/mac/anaconda3/lib/python3.11/site-packages/pandas/core/arrays/masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (
/Users/mac/anaconda3/lib/python3.11/site-packages/pandas/core/arrays/masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


Best Params: {'clf__C': 10.0}
CV F1 Mean : 0.9991
Test F1    : 0.9993

>>> Running model: RandomForest
Best Params: {'clf__max_depth': None, 'clf__n_estimators': 100}
CV F1 Mean : 0.9989
Test F1    : 0.9987

>>> Running model: Stacking


/Users/mac/anaconda3/lib/python3.11/site-packages/pandas/core/arrays/masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


#### ============================================================
#### STEP 7 - Model comparison (F1)
#### IMPORTANT: choose model by CV_F1_Mean, not Test_F1.
#### ============================================================

In [ ]:
print_section("STEP 7 - Model Comparison (Based on CV F1)")

results_df = pd.DataFrame(results)

results_df_nonbaseline = results_df[results_df["Model"] != "Baseline_Dummy"].copy()
results_df_nonbaseline = results_df_nonbaseline.sort_values(
    by="CV_F1_Mean",
    ascending=False
).reset_index(drop=True)

print("All model results:")
print(results_df)

print("\nRanked models by CV F1:")
print(results_df_nonbaseline)

results_df.to_csv(os.path.join(OUTPUT_DIR, "all_model_results.csv"), index=False)
results_df_nonbaseline.to_csv(os.path.join(OUTPUT_DIR, "ranked_model_results.csv"), index=False)


#### ============================================================
#### STEP 8 - Retrain best model on full training data
#### GridSearchCV(refit=True) already does this on the whole train set.
#### ============================================================

In [ ]:
print_section("STEP 8 - Retrain Best Model on Full Training Data")

best_model_name = results_df_nonbaseline.iloc[0]["Model"]
best_model = best_estimators[best_model_name]

print("Best model selected by CV F1:", best_model_name)
print("\nBest model object:")
print(best_model)

#### ============================================================
#### STEP 9 - Final evaluation on test set
#### Required metrics: F1, AUC, ROC, Accuracy
#### ============================================================

In [ ]:
print_section("STEP 9 - Final Evaluation on Test Set")

y_test_pred = best_model.predict(X_test)
y_test_score = get_model_scores(best_model, X_test)

final_accuracy = accuracy_score(y_test, y_test_pred)
final_precision = precision_score(y_test, y_test_pred, zero_division=0)
final_recall = recall_score(y_test, y_test_pred, zero_division=0)
final_f1 = f1_score(y_test, y_test_pred, zero_division=0)

print("Final Test Accuracy :", round(final_accuracy, 4))
print("Final Test Precision:", round(final_precision, 4))
print("Final Test Recall   :", round(final_recall, 4))
print("Final Test F1       :", round(final_f1, 4))

report = classification_report(
    y_test,
    y_test_pred,
    target_names=["Fake", "True"],
    digits=4
)

print("\nClassification Report:")
print(report)
save_text_report(report, "final_classification_report.txt")

# AUC / ROC
if y_test_score is not None:
    final_auc = roc_auc_score(y_test, y_test_score)
    fpr, tpr, thresholds = roc_curve(y_test, y_test_score)

    print("Final Test AUC      :", round(final_auc, 4))

    plt.figure(figsize=(7, 5))
    plt.plot(fpr, tpr, label=f"{best_model_name} (AUC={final_auc:.4f})")
    plt.plot([0, 1], [0, 1], linestyle="--")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title("ROC Curve - Final Model")
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "roc_curve_final_model.png"), dpi=300)
    plt.close()
else:
    final_auc = None
    print("AUC/ROC not available for this model.")

# Confusion Matrix
cm = confusion_matrix(y_test, y_test_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Fake", "True"])
fig, ax = plt.subplots(figsize=(6, 5))
disp.plot(ax=ax, cmap="Blues", colorbar=False)
plt.title(f"Confusion Matrix - {best_model_name}")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "confusion_matrix_final_model.png"), dpi=300)
plt.close()

#### ============================================================
#### Statistical component - Bootstrap confidence intervals
#### ============================================================

In [ ]:
print_section("STATISTICAL COMPONENT - Bootstrap Confidence Intervals")

acc_mean, acc_low, acc_high = bootstrap_metric_ci(
    y_test,
    y_test_pred,
    accuracy_score,
    n_rounds=BOOTSTRAP_ROUNDS,
    random_state=RANDOM_STATE
)

f1_mean, f1_low, f1_high = bootstrap_metric_ci(
    y_test,
    y_test_pred,
    lambda yt, yp: f1_score(yt, yp, zero_division=0),
    n_rounds=BOOTSTRAP_ROUNDS,
    random_state=RANDOM_STATE
)

bootstrap_report = f"""
Best Model: {best_model_name}

Bootstrap {BOOTSTRAP_ROUNDS} rounds - 95% Confidence Intervals

Accuracy : mean={acc_mean:.4f}, 95% CI=({acc_low:.4f}, {acc_high:.4f})
F1 Score : mean={f1_mean:.4f}, 95% CI=({f1_low:.4f}, {f1_high:.4f})
"""

print(bootstrap_report)
save_text_report(bootstrap_report, "bootstrap_confidence_intervals.txt")


#### ============================================================
#### Interpretability analysis
#### For interpretation, use Logistic Regression coefficients.
#### If final model is not LogisticRegression, fit a separate LR model.
#### ============================================================

In [ ]:
print_section("INTERPRETABILITY ANALYSIS")

if best_model_name == "LogisticRegression":
    interpret_model = best_model
else:
    interpret_model = Pipeline([
        ("tfidf", tfidf),
        ("clf", LogisticRegression(max_iter=3000, random_state=RANDOM_STATE))
    ])
    interpret_model.fit(X_train, y_train)

vectorizer = interpret_model.named_steps["tfidf"]
clf = interpret_model.named_steps["clf"]

feature_names = np.array(vectorizer.get_feature_names_out())
coef = clf.coef_[0]

top_true_idx = np.argsort(coef)[-20:][::-1]
top_fake_idx = np.argsort(coef)[:20]

interpret_lines = []
interpret_lines.append("Top terms indicating TRUE news:\n")
for idx in top_true_idx:
    interpret_lines.append(f"{feature_names[idx]:<30} {coef[idx]:.6f}")

interpret_lines.append("\nTop terms indicating FAKE news:\n")
for idx in top_fake_idx:
    interpret_lines.append(f"{feature_names[idx]:<30} {coef[idx]:.6f}")

interpret_text = "\n".join(interpret_lines)
print(interpret_text)
save_text_report(interpret_text, "top_terms_interpretability.txt")


#### ============================================================
#### Save final summary
#### ============================================================

In [ ]:
print_section("SAVE FINAL SUMMARY")

summary_lines = []
summary_lines.append("Fake News Detection - Final Summary V4")
summary_lines.append("=" * 70)
summary_lines.append(f"Training samples: {len(X_train)}")
summary_lines.append(f"Testing samples : {len(X_test)}")
summary_lines.append("")
summary_lines.append("All model results:")
summary_lines.append(results_df.to_string(index=False))
summary_lines.append("")
summary_lines.append(f"Best model selected by CV F1: {best_model_name}")
summary_lines.append("")
summary_lines.append("Final test metrics:")
summary_lines.append(f"Accuracy : {final_accuracy:.4f}")
summary_lines.append(f"Precision: {final_precision:.4f}")
summary_lines.append(f"Recall   : {final_recall:.4f}")
summary_lines.append(f"F1 Score : {final_f1:.4f}")
if final_auc is not None:
    summary_lines.append(f"AUC      : {final_auc:.4f}")
summary_lines.append("")
summary_lines.append("Classification Report:")
summary_lines.append(report)
summary_lines.append("")
summary_lines.append("Bootstrap Confidence Intervals:")
summary_lines.append(bootstrap_report)

final_summary = "\n".join(summary_lines)
save_text_report(final_summary, "final_project_summary.txt")

print("All results saved to folder:", OUTPUT_DIR)
print("Done.")